# Vaani KWS — V2 DS-CNN Training & Evaluation
---
**Project:** SIH 2026 — Low Latency & Efficient Voice Activator for Edge Devices  
**Keyword:** "Vaani"  
**Version:** V2  

## Prerequisites
Run `01_prepare_dataset_v2.ipynb` first to populate:
- `model_v2/data/train/` (positive, negative_silence, negative_background, negative_speech_commands)
- `model_v2/data/validation/`
- `model_v2/data/test/`

**Notebook 01 does NOT create features.** This notebook handles everything from WAV files onward.

## What This Notebook Does
1. Loads WAV dataset from `model_v2/data/`
2. Augments TRAINING data only (saves to `model_v2/data/train_augmented/`)
3. Extracts V1-compatible Log-Mel features from all WAVs
4. Saves features as `.npz` with category metadata
5. Builds the V1-baseline DS-CNN-Small architecture (~5.3K params)
6. Uses **class weights** to handle intentional class imbalance
7. Trains with Adam + EarlyStopping + ReduceLROnPlateau
8. Evaluates on untouched V2 test set
9. **Per-category false-activation analysis** (using saved metadata, not filename matching)
10. Multi-threshold evaluation (0.50 → 0.99)
11. V1 vs V2 comparison — same architecture, same frontend, isolating dataset effect

**This notebook does NOT read from model_v1 during training.**  
**This notebook does NOT perform quantization or TFLite conversion.**

## 1. Configuration & Imports

In [1]:
import os
import sys
import json
import shutil
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import tensorflow as tf
from tqdm import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Imports complete.")
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy:      {np.__version__}")
print(f"Librosa:    {librosa.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

Imports complete.
TensorFlow: 2.21.0
NumPy:      2.2.6
Librosa:    0.11.0
GPU available: []


## 2. Project Paths

In [2]:
# ============================================================
# PROJECT PATHS
# ============================================================

def find_project_root():
    """Walk up from notebook location to find project root."""
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "dataset").is_dir() and (candidate / "model_v1").is_dir():
            return candidate
    fallback = Path(r"c:/Users/Mayank Singh/Codes/SIH 2026")
    if fallback.is_dir():
        return fallback
    raise FileNotFoundError("Could not find PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
MODEL_V2 = PROJECT_ROOT / "model_v2"

DATA_DIR       = MODEL_V2 / "data"
FEATURE_DIR    = MODEL_V2 / "features"
CHECKPOINT_DIR = MODEL_V2 / "checkpoints"
EVALUATION_DIR = MODEL_V2 / "evaluation"
EXPORT_DIR     = MODEL_V2 / "exports"
AUG_DIR        = DATA_DIR / "train_augmented"
TEMP_DIR       = PROJECT_ROOT / "temp"

for d in [FEATURE_DIR, CHECKPOINT_DIR, EVALUATION_DIR, EXPORT_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# V1 metrics path (for comparison later — NOT used for training)
V1_METRICS_PATH = PROJECT_ROOT / "model_v1" / "evaluation" / "test_metrics.json"
V1_THRESHOLD_PATH = PROJECT_ROOT / "model_v1" / "evaluation" / "threshold_metrics.json"

print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"MODEL_V2:      {MODEL_V2}")
print(f"DATA_DIR:      {DATA_DIR}")
print(f"FEATURE_DIR:   {FEATURE_DIR}")
print(f"AUG_DIR:       {AUG_DIR}")

PROJECT_ROOT:  c:\Users\Mayank Singh\Codes\SIH 2026
MODEL_V2:      c:\Users\Mayank Singh\Codes\SIH 2026\model_v2
DATA_DIR:      c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\data
FEATURE_DIR:   c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\features
AUG_DIR:       c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\data\train_augmented


## 3. Inspect Dataset from model_v2/data/

In [3]:
# ============================================================
# INSPECT DATASET
# ============================================================

SPLITS = ["train", "validation", "test"]
CATEGORIES = ["positive", "negative_silence", "negative_background", "negative_speech_commands"]

print("=" * 60)
print("DATASET INSPECTION")
print("=" * 60)
print()

dataset_files = {}  # (split, category) -> list of Path

for split in SPLITS:
    for cat in CATEGORIES:
        d = DATA_DIR / split / cat
        if d.is_dir():
            files = sorted(d.glob("*.wav"))
        else:
            files = []
        dataset_files[(split, cat)] = files

# Print summary table
print(f"{'Split':12s} {'Category':30s} {'Count':>7s}")
print("─" * 55)
for split in SPLITS:
    split_total = 0
    for cat in CATEGORIES:
        n = len(dataset_files[(split, cat)])
        split_total += n
        print(f"  {split:10s} {cat:30s} {n:>7d}")
    print(f"  {split:10s} {'TOTAL':30s} {split_total:>7d}")
    print()

# Check for empty categories
print("Warnings:")
found_warning = False
for split in SPLITS:
    for cat in CATEGORIES:
        n = len(dataset_files[(split, cat)])
        if n == 0:
            print(f"  ⚠ {split}/{cat}: 0 files")
            if cat == "negative_silence" and split in ["validation", "test"]:
                print(f"    → Expected: silence clips are training-only (single source recording).")
            found_warning = True
if not found_warning:
    print("  None.")

DATASET INSPECTION

Split        Category                         Count
───────────────────────────────────────────────────────
  train      positive                           322
  train      negative_silence                   307
  train      negative_background                551
  train      negative_speech_commands          2449
  train      TOTAL                             3629

  validation positive                            24
  validation negative_silence                     0
  validation negative_background                121
  validation negative_speech_commands           523
  validation TOTAL                              668

  test       positive                             6
  test       negative_silence                     0
  test       negative_background                119
  test       negative_speech_commands           524
  test       TOTAL                              649

Warnings:
  ⚠ validation/negative_silence: 0 files
    → Expected: silence clips are trai

## 4. Training Augmentation

**Rules:**
- Only TRAIN data is augmented
- Validation/test remain untouched — NEVER augmented
- Original WAV files are NEVER modified
- Augmented copies saved under `model_v2/data/train_augmented/`
- 2 augmented versions per original training sample
- Silence/background: conservative augmentation only (gain + mild noise)
- Deterministic seed for reproducibility

**Augmentation types:**
1. Random gain (±6 dB)
2. Random time shift (±80ms)
3. Additive Gaussian noise (SNR 10–25 dB)
4. Mild pitch shift (±1 semitone)
5. Mild artificial reverb

In [4]:
# ============================================================
# AUGMENTATION FUNCTIONS
# ============================================================

SAMPLE_RATE = 16000

def augment_audio(audio, sr, rng, aug_type):
    """Apply a single augmentation. Returns augmented audio."""

    if aug_type == "gain":
        gain_db = rng.uniform(-6, 6)
        gain_linear = 10 ** (gain_db / 20)
        return audio * gain_linear

    elif aug_type == "time_shift":
        max_shift = int(0.08 * sr)  # ±80ms
        shift = rng.randint(-max_shift, max_shift + 1)
        augmented = np.zeros_like(audio)
        if shift > 0:
            augmented[shift:] = audio[:-shift] if shift < len(audio) else 0
        elif shift < 0:
            augmented[:shift] = audio[-shift:]
        else:
            augmented = audio.copy()
        return augmented

    elif aug_type == "noise":
        snr_db = rng.uniform(10, 25)
        signal_power = np.mean(audio ** 2)
        if signal_power > 0:
            noise_power = signal_power / (10 ** (snr_db / 10))
            noise = rng.normal(0, np.sqrt(noise_power), len(audio)).astype(np.float32)
            return audio + noise
        return audio

    elif aug_type == "pitch_shift":
        n_steps = rng.uniform(-1, 1)
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)

    elif aug_type == "reverb":
        ir_len = int(0.05 * sr)  # 50ms reverb tail
        ir = np.exp(-np.linspace(0, 5, ir_len)).astype(np.float32)
        ir = ir * rng.uniform(0.1, 0.3)
        ir[0] = 1.0
        ir = ir / np.sum(ir)
        augmented = np.convolve(audio, ir, mode='same').astype(np.float32)
        return augmented

    return audio

def create_augmented_file(src_path, dst_path, sr, rng, aug_type):
    """Load, augment, and save a WAV file."""
    audio, _ = librosa.load(str(src_path), sr=sr, mono=True)

    # Ensure 1-second length
    target_len = sr
    if len(audio) < target_len:
        padded = np.zeros(target_len, dtype=np.float32)
        start = (target_len - len(audio)) // 2
        padded[start:start + len(audio)] = audio
        audio = padded
    elif len(audio) > target_len:
        start = (len(audio) - target_len) // 2
        audio = audio[start:start + target_len]

    augmented = augment_audio(audio, sr, rng, aug_type)
    augmented = np.clip(augmented, -1.0, 1.0)
    sf.write(str(dst_path), augmented, sr, subtype='PCM_16')

print("Augmentation functions defined.")

Augmentation functions defined.


In [5]:
# ============================================================
# PERFORM AUGMENTATION — TRAINING ONLY
# ============================================================

AUG_PER_SAMPLE = 2
# Full augmentation types for positive and speech commands
AUG_TYPES_FULL = ["gain", "time_shift", "noise", "pitch_shift", "reverb"]
# Conservative augmentation for silence/background
AUG_TYPES_CONSERVATIVE = ["gain", "noise"]

rng_aug = np.random.RandomState(SEED + 100)

# Create augmentation directories
for cat in CATEGORIES:
    (AUG_DIR / cat).mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("AUGMENTATION — TRAINING SET ONLY")
print("=" * 60)
print(f"Augmented copies per sample: {AUG_PER_SAMPLE}")
print(f"Full augmentations: {AUG_TYPES_FULL}")
print(f"Conservative (silence/bg): {AUG_TYPES_CONSERVATIVE}")
print()

aug_manifest = []
aug_count = 0
aug_errors = []

for cat in CATEGORIES:
    train_files = dataset_files[("train", cat)]
    if not train_files:
        print(f"  {cat}: 0 training files — skipping augmentation")
        continue

    # Choose augmentation strategy
    is_conservative = cat in ("negative_silence", "negative_background")
    aug_types = AUG_TYPES_CONSERVATIVE if is_conservative else AUG_TYPES_FULL
    strategy_name = "conservative" if is_conservative else "full"

    print(f"  {cat}: {len(train_files)} files × {AUG_PER_SAMPLE} = "
          f"{len(train_files) * AUG_PER_SAMPLE} augmented copies ({strategy_name})")

    for f in tqdm(train_files, desc=f"  Aug {cat}", leave=True):
        for aug_i in range(AUG_PER_SAMPLE):
            aug_type = rng_aug.choice(aug_types)
            aug_filename = f"aug_{aug_type}_{aug_i}_{f.name}"
            dst_path = AUG_DIR / cat / aug_filename

            try:
                create_augmented_file(f, dst_path, SAMPLE_RATE, rng_aug, aug_type)
                aug_manifest.append({
                    "filepath": str(dst_path),
                    "original": str(f),
                    "category": cat,
                    "aug_type": aug_type,
                    "aug_index": aug_i,
                })
                aug_count += 1
            except Exception as e:
                aug_errors.append(f"{f.name}: {e}")

print(f"\nTotal augmented files created: {aug_count}")
if aug_errors:
    print(f"Augmentation errors: {len(aug_errors)}")
    for err in aug_errors[:5]:
        print(f"  {err}")
else:
    print("✓ No augmentation errors.")

# Save augmentation manifest
aug_manifest_path = DATA_DIR / "manifests" / "augmentation_manifest.csv"
aug_manifest_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(aug_manifest).to_csv(aug_manifest_path, index=False)
print(f"Saved augmentation manifest: {aug_manifest_path}")

AUGMENTATION — TRAINING SET ONLY
Augmented copies per sample: 2
Full augmentations: ['gain', 'time_shift', 'noise', 'pitch_shift', 'reverb']
Conservative (silence/bg): ['gain', 'noise']

  positive: 322 files × 2 = 644 augmented copies (full)


  Aug positive:   0%|          | 0/322 [00:00<?, ?it/s]c:\Users\Mayank Singh\Codes\SIH 2026\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  Aug positive: 100%|██████████| 322/322 [00:08<00:00, 37.10it/s]


  negative_silence: 307 files × 2 = 614 augmented copies (conservative)


  Aug negative_silence: 100%|██████████| 307/307 [00:02<00:00, 122.07it/s]


  negative_background: 551 files × 2 = 1102 augmented copies (conservative)


  Aug negative_background: 100%|██████████| 551/551 [00:05<00:00, 97.56it/s] 


  negative_speech_commands: 2449 files × 2 = 4898 augmented copies (full)


  Aug negative_speech_commands: 100%|██████████| 2449/2449 [00:28<00:00, 86.43it/s]


Total augmented files created: 7258
✓ No augmentation errors.
Saved augmentation manifest: c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\data\manifests\augmentation_manifest.csv


## 5. Log-Mel Feature Extraction (V1-Compatible)

Using the **exact same** V1 frontend:
- Sample rate: 16,000 Hz
- Window: 30 ms (480 samples)
- Hop: 20 ms (320 samples)
- n_fft: 480
- n_mels: 40
- fmin: 20 Hz, fmax: 7,600 Hz
- center: False
- window: Hann
- power: 2.0
- power_to_db(ref=np.max)

Normalization (applied later before training):
- `(x + 80.0) / 80.0`, clipped to [0, 1]

In [6]:
# ============================================================
# FEATURE EXTRACTION CONFIGURATION (V1-compatible)
# ============================================================

WINDOW_MS = 30
HOP_MS = 20
N_FFT = 480
N_MELS = 40
FMIN = 20
FMAX = 7600

WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_MS / 1000)  # 480
HOP_SAMPLES = int(SAMPLE_RATE * HOP_MS / 1000)         # 320

print("Feature extraction configuration (V1-compatible):")
print(f"  Sample rate:    {SAMPLE_RATE} Hz")
print(f"  Window:         {WINDOW_MS} ms ({WINDOW_SAMPLES} samples)")
print(f"  Hop:            {HOP_MS} ms ({HOP_SAMPLES} samples)")
print(f"  n_fft:          {N_FFT}")
print(f"  n_mels:         {N_MELS}")
print(f"  fmin:           {FMIN} Hz")
print(f"  fmax:           {FMAX} Hz")
print(f"  center:         False")
print(f"  window:         Hann")
print(f"  power:          2.0")

def extract_log_mel(filepath):
    """Extract log-Mel spectrogram. V1-compatible."""
    audio, sr = librosa.load(str(filepath), sr=SAMPLE_RATE, mono=True)

    # Ensure exactly 1 second
    target_samples = SAMPLE_RATE
    if len(audio) < target_samples:
        padded = np.zeros(target_samples, dtype=np.float32)
        start = (target_samples - len(audio)) // 2
        padded[start:start + len(audio)] = audio
        audio = padded
    elif len(audio) > target_samples:
        start = (len(audio) - target_samples) // 2
        audio = audio[start:start + target_samples]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_SAMPLES, win_length=WINDOW_SAMPLES,
        window="hann", center=False,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
        power=2.0,
    )

    log_mel = librosa.power_to_db(mel, ref=np.max)
    return log_mel.astype(np.float32)

# Determine feature shape dynamically from one file
_test_files = dataset_files[("train", "positive")]
if not _test_files:
    _test_files = [f for k, v in dataset_files.items() for f in v if v]
_test_feature = extract_log_mel(_test_files[0])
FEATURE_SHAPE = _test_feature.shape
print(f"\nDerived feature shape: {FEATURE_SHAPE} (n_mels × time_frames)")

Feature extraction configuration (V1-compatible):
  Sample rate:    16000 Hz
  Window:         30 ms (480 samples)
  Hop:            20 ms (320 samples)
  n_fft:          480
  n_mels:         40
  fmin:           20 Hz
  fmax:           7600 Hz
  center:         False
  window:         Hann
  power:          2.0

Derived feature shape: (40, 49) (n_mels × time_frames)


In [7]:
# ============================================================
# EXTRACT FEATURES FOR ALL SPLITS
# ============================================================

def collect_files_for_split(split):
    """Collect WAV files for a split. Training includes augmented files."""
    file_list = []  # (filepath, label, category)

    for cat in CATEGORIES:
        label = 1 if cat == "positive" else 0

        # Original files
        for f in dataset_files[(split, cat)]:
            file_list.append((f, label, cat))

        # Augmented files — training split ONLY
        if split == "train":
            aug_cat_dir = AUG_DIR / cat
            if aug_cat_dir.is_dir():
                for f in sorted(aug_cat_dir.glob("*.wav")):
                    file_list.append((f, label, cat))

    return file_list

def extract_features_for_split(split_name):
    """Extract features for all files in a split."""
    file_list = collect_files_for_split(split_name)

    print(f"\n{'=' * 60}")
    print(f"EXTRACTING FEATURES: {split_name.upper()}")
    print(f"{'=' * 60}")
    print(f"Files: {len(file_list)}")

    X_list = []
    y_list = []
    paths_list = []
    categories_list = []
    errors = []

    for filepath, label, category in tqdm(file_list, desc=split_name):
        try:
            feature = extract_log_mel(filepath)
            X_list.append(feature)
            y_list.append(label)
            paths_list.append(str(filepath))
            categories_list.append(category)
        except Exception as e:
            errors.append(f"{filepath}: {e}")

    X = np.stack(X_list)
    y = np.array(y_list, dtype=np.int64)
    paths = np.array(paths_list)
    categories = np.array(categories_list)

    print(f"Feature array shape: {X.shape}")
    print(f"Labels shape:        {y.shape}")
    print(f"Value range:         [{X.min():.2f}, {X.max():.2f}]")
    print(f"Label distribution:  0={np.sum(y==0)}, 1={np.sum(y==1)}")
    print(f"Category counts:")
    for cat_name, cnt in sorted(Counter(categories).items()):
        print(f"  {cat_name}: {cnt}")

    if errors:
        print(f"Errors: {len(errors)}")
        for e in errors[:5]:
            print(f"  {e}")

    return X, y, paths, categories

X_train_raw, y_train, paths_train, cats_train = extract_features_for_split("train")
X_val_raw, y_val, paths_val, cats_val = extract_features_for_split("validation")
X_test_raw, y_test, paths_test, cats_test = extract_features_for_split("test")


EXTRACTING FEATURES: TRAIN
Files: 10887


train: 100%|██████████| 10887/10887 [01:15<00:00, 144.01it/s]


Feature array shape: (10887, 40, 49)
Labels shape:        (10887,)
Value range:         [-80.00, 0.00]
Label distribution:  0=9921, 1=966
Category counts:
  negative_background: 1653
  negative_silence: 921
  negative_speech_commands: 7347
  positive: 966

EXTRACTING FEATURES: VALIDATION
Files: 668


validation: 100%|██████████| 668/668 [00:04<00:00, 151.87it/s]


Feature array shape: (668, 40, 49)
Labels shape:        (668,)
Value range:         [-80.00, 0.00]
Label distribution:  0=644, 1=24
Category counts:
  negative_background: 121
  negative_speech_commands: 523
  positive: 24

EXTRACTING FEATURES: TEST
Files: 649


test: 100%|██████████| 649/649 [00:04<00:00, 148.44it/s]

Feature array shape: (649, 40, 49)
Labels shape:        (649,)
Value range:         [-80.00, 0.00]
Label distribution:  0=643, 1=6
Category counts:
  negative_background: 119
  negative_speech_commands: 524
  positive: 6


## 6. Save Features as .npz

Each file contains: `X`, `y`, `paths`, `categories`  
Category metadata enables per-category evaluation without fragile filename matching.

In [8]:
# ============================================================
# SAVE FEATURES
# ============================================================

for split_name, X, y, paths, cats in [
    ("train", X_train_raw, y_train, paths_train, cats_train),
    ("validation", X_val_raw, y_val, paths_val, cats_val),
    ("test", X_test_raw, y_test, paths_test, cats_test),
]:
    output_path = FEATURE_DIR / f"{split_name}.npz"
    np.savez_compressed(
        str(output_path),
        X=X, y=y, paths=paths, categories=cats,
    )
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"Saved: {output_path}")
    print(f"  X shape: {X.shape}, dtype: {X.dtype}")
    print(f"  y shape: {y.shape}, categories: {len(np.unique(cats))}")
    print(f"  File size: {file_size_mb:.1f} MB")

print(f"\nFeature shape (derived dynamically): {X_train_raw.shape[1:]}")

Saved: c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\features\train.npz
  X shape: (10887, 40, 49), dtype: float32
  y shape: (10887,), categories: 4
  File size: 64.6 MB
Saved: c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\features\validation.npz
  X shape: (668, 40, 49), dtype: float32
  y shape: (668,), categories: 3
  File size: 3.8 MB
Saved: c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\features\test.npz
  X shape: (649, 40, 49), dtype: float32
  y shape: (649,), categories: 3
  File size: 3.6 MB

Feature shape (derived dynamically): (40, 49)


## 7. Pre-Training Sanity Checks

In [9]:
# ============================================================
# PRE-TRAINING SANITY CHECKS
# ============================================================

print("=" * 60)
print("PRE-TRAINING SANITY CHECKS")
print("=" * 60)

checks_passed = 0
checks_total = 0

# Check 1: Feature shapes consistent
checks_total += 1
assert X_train_raw.shape[1:] == X_val_raw.shape[1:] == X_test_raw.shape[1:], \
    f"Inconsistent shapes: train={X_train_raw.shape[1:]}, val={X_val_raw.shape[1:]}, test={X_test_raw.shape[1:]}"
checks_passed += 1
print(f"✓ Check 1: Feature shapes consistent: {X_train_raw.shape[1:]}")

# Check 2: Labels are binary
checks_total += 1
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    unique = set(np.unique(y))
    assert unique.issubset({0, 1}), f"Unexpected labels in {name}: {unique}"
checks_passed += 1
print(f"✓ Check 2: Labels are binary {{0, 1}}")

# Check 3: No NaN/Inf
checks_total += 1
for name, X in [("train", X_train_raw), ("val", X_val_raw), ("test", X_test_raw)]:
    assert not np.any(np.isnan(X)), f"NaN found in {name}"
    assert not np.any(np.isinf(X)), f"Inf found in {name}"
checks_passed += 1
print(f"✓ Check 3: No NaN/Inf in features")

# Check 4: No augmented files in val/test
checks_total += 1
for name, paths in [("val", paths_val), ("test", paths_test)]:
    aug_in_split = sum(1 for p in paths if "train_augmented" in str(p))
    assert aug_in_split == 0, f"Found {aug_in_split} augmented files in {name}!"
checks_passed += 1
print(f"✓ Check 4: No augmented files in validation/test")

# Check 5: Value range is reasonable (log-mel dB)
checks_total += 1
for name, X in [("train", X_train_raw), ("val", X_val_raw), ("test", X_test_raw)]:
    assert X.min() >= -81, f"{name} min too low: {X.min()}"
    assert X.max() <= 1, f"{name} max too high: {X.max()}"
checks_passed += 1
print(f"✓ Check 5: Feature values in expected dB range")

# Check 6: Category metadata matches file count
checks_total += 1
for name, cats in [("train", cats_train), ("val", cats_val), ("test", cats_test)]:
    for cat_val in np.unique(cats):
        assert cat_val in CATEGORIES, f"Unknown category in {name}: {cat_val}"
checks_passed += 1
print(f"✓ Check 6: Category metadata contains valid categories")

# Check 7: No duplicate paths
checks_total += 1
for name, paths in [("train", paths_train), ("val", paths_val), ("test", paths_test)]:
    unique_paths = len(set(paths))
    assert unique_paths == len(paths), f"Duplicate paths in {name}: {len(paths) - unique_paths}"
checks_passed += 1
print(f"✓ Check 7: No duplicate feature paths")

# Check 8: train counts match expected (original + augmented)
checks_total += 1
n_orig_train = sum(len(dataset_files[("train", cat)]) for cat in CATEGORIES)
n_aug_files = sum(1 for p in paths_train if "train_augmented" in str(p))
n_total_expected = n_orig_train + n_aug_files
# Allow for augmentation errors
assert abs(len(y_train) - (n_orig_train + aug_count)) <= len(aug_errors), \
    f"Train count mismatch: {len(y_train)} features vs {n_orig_train} original + {aug_count} augmented"
checks_passed += 1
print(f"✓ Check 8: Train count = {n_orig_train} original + {aug_count} augmented = {len(y_train)} total")

print(f"\n{'=' * 60}")
print(f"SANITY CHECKS: {checks_passed}/{checks_total} PASSED")
print(f"{'=' * 60}")

PRE-TRAINING SANITY CHECKS
✓ Check 1: Feature shapes consistent: (40, 49)
✓ Check 2: Labels are binary {0, 1}
✓ Check 3: No NaN/Inf in features
✓ Check 4: No augmented files in validation/test
✓ Check 5: Feature values in expected dB range
✓ Check 6: Category metadata contains valid categories
✓ Check 7: No duplicate feature paths
✓ Check 8: Train count = 3629 original + 7258 augmented = 10887 total

SANITY CHECKS: 8/8 PASSED


## 8. Normalize & Prepare Data

In [10]:
# ============================================================
# PREPARE DATA — Normalize + Add Channel Dimension
# ============================================================
# V1 convention: normalize [-80, 0] dB → [0, 1]
# Formula: (x + 80.0) / 80.0, then clip to [0, 1]

# Add channel dimension: (N, n_mels, time) → (N, n_mels, time, 1)
X_train = X_train_raw[..., np.newaxis]
X_val   = X_val_raw[..., np.newaxis]
X_test  = X_test_raw[..., np.newaxis]

# Normalize
X_train = np.clip((X_train + 80.0) / 80.0, 0.0, 1.0)
X_val   = np.clip((X_val + 80.0) / 80.0, 0.0, 1.0)
X_test  = np.clip((X_test + 80.0) / 80.0, 0.0, 1.0)

# DYNAMIC INPUT SHAPE — derived from data, NEVER hardcoded
INPUT_SHAPE = X_train.shape[1:]  # e.g., (40, 49, 1)
NUM_CLASSES = 2

print("=" * 60)
print("PREPARED DATA")
print("=" * 60)
print(f"\nINPUT_SHAPE (derived dynamically): {INPUT_SHAPE}")
print(f"NUM_CLASSES: {NUM_CLASSES}")
print(f"\nPrepared shapes:")
print(f"  Train:      {X_train.shape}")
print(f"  Validation: {X_val.shape}")
print(f"  Test:       {X_test.shape}")
print(f"\nNormalized value range:")
print(f"  Train:  [{X_train.min():.4f}, {X_train.max():.4f}]")
print(f"  Val:    [{X_val.min():.4f}, {X_val.max():.4f}]")
print(f"  Test:   [{X_test.min():.4f}, {X_test.max():.4f}]")

print(f"\nLabel distribution:")
for name, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    n0 = np.sum(y == 0)
    n1 = np.sum(y == 1)
    print(f"  {name:6s}: negative={n0:>5,}, positive(vaani)={n1:>5,}, total={n0+n1:>5,}")

PREPARED DATA

INPUT_SHAPE (derived dynamically): (40, 49, 1)
NUM_CLASSES: 2

Prepared shapes:
  Train:      (10887, 40, 49, 1)
  Validation: (668, 40, 49, 1)
  Test:       (649, 40, 49, 1)

Normalized value range:
  Train:  [0.0000, 1.0000]
  Val:    [0.0000, 1.0000]
  Test:   [0.0000, 1.0000]

Label distribution:
  Train : negative=9,921, positive(vaani)=  966, total=10,887
  Val   : negative=  644, positive(vaani)=   24, total=  668
  Test  : negative=  643, positive(vaani)=    6, total=  649


## 9. Class Weights

In [11]:
# ============================================================
# CLASS WEIGHTS — Handle Intentional Imbalance
# ============================================================
# V2 intentionally has more negatives than positives to preserve
# negative diversity (silence, background, speech commands).
#
# Formula: class_weight[c] = total / (num_classes * count[c])
# This is sklearn's "balanced" formula.

n_total = len(y_train)
n_classes = NUM_CLASSES
class_counts = np.bincount(y_train, minlength=n_classes)

class_weights = {}
for c in range(n_classes):
    if class_counts[c] > 0:
        class_weights[c] = n_total / (n_classes * class_counts[c])
    else:
        class_weights[c] = 1.0

print("=" * 60)
print("CLASS WEIGHTS")
print("=" * 60)
print(f"\nFormula: weight[c] = N_total / (N_classes × N_c)")
print(f"  N_total   = {n_total:,}")
print(f"  N_classes = {n_classes}")
print()
for c in range(n_classes):
    label_name = "negative" if c == 0 else "vaani"
    print(f"  Class {c} ({label_name:8s}): count = {class_counts[c]:>6,}, weight = {class_weights[c]:.4f}")

print(f"\nThese weights will be passed to model.fit(class_weight=...).")
print(f"This ensures the model doesn't simply learn to predict 'negative' for everything.")

CLASS WEIGHTS

Formula: weight[c] = N_total / (N_classes × N_c)
  N_total   = 10,887
  N_classes = 2

  Class 0 (negative): count =  9,921, weight = 0.5487
  Class 1 (vaani   ): count =    966, weight = 5.6351

These weights will be passed to model.fit(class_weight=...).
This ensures the model doesn't simply learn to predict 'negative' for everything.


## 10. DS-CNN-Small Architecture (V1 Baseline, Unchanged)

In [12]:
# ============================================================
# DS-CNN-SMALL ARCHITECTURE (V1 Baseline)
# ============================================================
# Architecture:
#   Input
#   → Conv2D 16 filters, 3×3, stride 2
#   → BatchNorm → ReLU
#   → DS block (16 filters)
#   → MaxPool 2×2
#   → DS block (24 filters)
#   → MaxPool 2×2
#   → DS block (32 filters)
#   → DS block (32 filters)
#   → Global Average Pooling
#   → Dense 32 → ReLU
#   → Dropout 0.2
#   → Dense 2 → Softmax
#
# Target: ~5.3K parameters
# DO NOT silently change the architecture.

def ds_cnn_block(x, filters, name):
    """Depthwise-separable CNN block."""
    # Depthwise convolution
    x = tf.keras.layers.DepthwiseConv2D(
        kernel_size=(3, 3),
        padding="same",
        use_bias=False,
        name=f"{name}_depthwise"
    )(x)
    x = tf.keras.layers.BatchNormalization(name=f"{name}_dw_bn")(x)
    x = tf.keras.layers.ReLU(name=f"{name}_dw_relu")(x)

    # Pointwise convolution
    x = tf.keras.layers.Conv2D(
        filters=filters,
        kernel_size=(1, 1),
        padding="same",
        use_bias=False,
        name=f"{name}_pointwise"
    )(x)
    x = tf.keras.layers.BatchNormalization(name=f"{name}_pw_bn")(x)
    x = tf.keras.layers.ReLU(name=f"{name}_pw_relu")(x)

    return x

def build_model(input_shape):
    """Build DS-CNN-Small. Input shape derived dynamically."""

    inputs = tf.keras.Input(shape=input_shape, name="log_mel_input")

    # Initial convolution
    x = tf.keras.layers.Conv2D(
        filters=16, kernel_size=(3, 3), strides=(2, 2),
        padding="same", use_bias=False, name="initial_conv"
    )(inputs)
    x = tf.keras.layers.BatchNormalization(name="initial_bn")(x)
    x = tf.keras.layers.ReLU(name="initial_relu")(x)

    # DS-CNN blocks
    x = ds_cnn_block(x, filters=16, name="ds_block_1")
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool_1")(x)

    x = ds_cnn_block(x, filters=24, name="ds_block_2")
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool_2")(x)

    x = ds_cnn_block(x, filters=32, name="ds_block_3")
    x = ds_cnn_block(x, filters=32, name="ds_block_4")

    # Global average pooling
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pool")(x)

    # Dense head
    x = tf.keras.layers.Dense(32, activation="relu", name="dense")(x)
    x = tf.keras.layers.Dropout(0.2, name="dropout")(x)

    outputs = tf.keras.layers.Dense(
        NUM_CLASSES, activation="softmax", name="classifier"
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Vaani_DS_CNN_Small_V2")
    return model

# Build model with DYNAMIC input shape
model = build_model(INPUT_SHAPE)
model.summary()

Model: "Vaani_DS_CNN_Small_V2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ log_mel_input (InputLayer)      │ (None, 40, 49, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ initial_conv (Conv2D)           │ (None, 20, 25, 16)     │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ initial_bn (BatchNormalization) │ (None, 20, 25, 16)     │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ initial_relu (ReLU)             │ (None, 20, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_1_depthwise            │ (None, 20, 25, 16)     │           144 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_1_dw_bn                │ (None, 20, 25, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_1_dw_relu (ReLU)       │ (None, 20, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_1_pointwise (Conv2D)   │ (None, 20, 25, 16)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_1_pw_bn                │ (None, 20, 25, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_1_pw_relu (ReLU)       │ (None, 20, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling2D)           │ (None, 10, 12, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_2_depthwise            │ (None, 10, 12, 16)     │           144 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_2_dw_bn                │ (None, 10, 12, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_2_dw_relu (ReLU)       │ (None, 10, 12, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_2_pointwise (Conv2D)   │ (None, 10, 12, 24)     │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_2_pw_bn                │ (None, 10, 12, 24)     │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_2_pw_relu (ReLU)       │ (None, 10, 12, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling2D)           │ (None, 5, 6, 24)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_3_depthwise            │ (None, 5, 6, 24)       │           216 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_3_dw_bn                │ (None, 5, 6, 24)       │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block_3_dw_relu (ReLU)       │ (None, 5, 6, 24)       │             

 Total params: 5,322 (20.79 KB)

 Trainable params: 4,906 (19.16 KB)

 Non-trainable params: 416 (1.62 KB)

In [13]:
# ============================================================
# MODEL SIZE
# ============================================================

trainable_params = np.sum([np.prod(v.shape) for v in model.trainable_variables])
non_trainable_params = np.sum([np.prod(v.shape) for v in model.non_trainable_variables])
total_params = trainable_params + non_trainable_params
fp32_size_kb = total_params * 4 / 1024

print("=" * 60)
print("MODEL SIZE")
print("=" * 60)
print(f"Trainable parameters:     {trainable_params:,}")
print(f"Non-trainable parameters: {non_trainable_params:,}")
print(f"Total parameters:         {total_params:,}")
print(f"Approx FP32 weights:      {fp32_size_kb:.2f} KB")
print(f"\nInput shape: {INPUT_SHAPE} (derived dynamically)")

MODEL SIZE
Trainable parameters:     4,906
Non-trainable parameters: 418
Total parameters:         5,324
Approx FP32 weights:      20.80 KB

Input shape: (40, 49, 1) (derived dynamically)


## 11. Compile & Train

In [14]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print(f"  Optimizer:          Adam")
print(f"  Learning rate:      {LEARNING_RATE}")
print(f"  Batch size:         {BATCH_SIZE}")
print(f"  Max epochs:         {EPOCHS}")
print(f"  Class weights:      {class_weights}")
print(f"  EarlyStopping:      patience=8, restore_best_weights=True")
print(f"  ReduceLROnPlateau:  factor=0.5, patience=3, min_lr=1e-6")
print(f"  ModelCheckpoint:    save_best_only=True, monitor=val_loss")

TRAINING CONFIGURATION
  Optimizer:          Adam
  Learning rate:      0.001
  Batch size:         32
  Max epochs:         50
  Class weights:      {0: np.float64(0.5486846084064106), 1: np.float64(5.635093167701863)}
  EarlyStopping:      patience=8, restore_best_weights=True
  ReduceLROnPlateau:  factor=0.5, patience=3, min_lr=1e-6
  ModelCheckpoint:    save_best_only=True, monitor=val_loss


In [15]:
# ============================================================
# COMPILE + CALLBACKS + TRAIN
# ============================================================

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

best_model_path = CHECKPOINT_DIR / "best_model_v2.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        str(EVALUATION_DIR / "training_history_v2.csv")
    ),
]

print()
print("=" * 60)
print("TRAINING")
print("=" * 60)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print()
print("Training complete.")
print(f"Best checkpoint saved: {best_model_path}")


TRAINING
Epoch 1/50
341/341 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7663 - loss: 0.5286
Epoch 1: val_loss improved from None to 0.17271, saving model to c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\checkpoints\best_model_v2.keras
341/341 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.8164 - loss: 0.4021 - val_accuracy: 0.9641 - val_loss: 0.1727 - learning_rate: 0.0010
Epoch 2/50
340/341 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9140 - loss: 0.2038
Epoch 2: val_loss improved from 0.17271 to 0.07524, saving model to c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\checkpoints\best_model_v2.keras
341/341 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9215 - loss: 0.1893 - val_accuracy: 0.9686 - val_loss: 0.0752 - learning_rate: 0.0010
Epoch 3/50
338/341 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9430 - loss: 0.1397
Epoch 3: val_loss did not improve from 0.07524
341/341 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9446 - loss: 0.1324 - val_accuracy: 0.8952 - val_loss: 0.33

## 12. Save Model & Config

In [16]:
# ============================================================
# SAVE FINAL/BEST MODEL
# ============================================================

final_model_path = EXPORT_DIR / "vaani_dscnn_v2.keras"
model.save(str(final_model_path))

print(f"Final model saved: {final_model_path}")
print(f"File size: {os.path.getsize(final_model_path) / 1024:.1f} KB")

# Save model config
model_config = {
    "model": "DS-CNN-Small",
    "version": "V2",
    "keyword": "Vaani",
    "input": {
        "sample_rate": SAMPLE_RATE,
        "window_ms": WINDOW_MS,
        "hop_ms": HOP_MS,
        "n_fft": N_FFT,
        "n_mels": N_MELS,
        "fmin": FMIN,
        "fmax": FMAX,
        "input_shape": list(INPUT_SHAPE),
    },
    "classes": {"0": "negative", "1": "vaani"},
    "training": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "class_weights": {str(k): float(v) for k, v in class_weights.items()},
        "augmentation_per_sample": AUG_PER_SAMPLE,
    },
    "parameters": int(total_params),
    "dataset": {
        "train_samples": int(len(y_train)),
        "train_original": int(n_orig_train) if 'n_orig_train' in dir() else "unknown",
        "train_augmented": int(aug_count),
        "val_samples": int(len(y_val)),
        "test_samples": int(len(y_test)),
    },
    "note": "V1 and V2 use identical DS-CNN architecture and feature frontend. "
            "This comparison isolates the effect of dataset changes.",
}

config_path = EVALUATION_DIR / "model_config_v2.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=4)
print(f"Config saved: {config_path}")

Final model saved: c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\exports\vaani_dscnn_v2.keras
File size: 204.0 KB
Config saved: c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\evaluation\model_config_v2.json


In [17]:
# ============================================================
# TRAINING CURVES (text-based summary)
# ============================================================

print("=" * 60)
print("TRAINING HISTORY")
print("=" * 60)

hist = history.history
n_epochs = len(hist['loss'])

print(f"\nTotal epochs trained: {n_epochs}")
print(f"\n{'Epoch':>5s} {'Loss':>10s} {'Acc':>10s} {'Val Loss':>10s} {'Val Acc':>10s} {'LR':>12s}")
print("─" * 60)

for i in range(n_epochs):
    lr = hist.get('lr', [LEARNING_RATE] * n_epochs)[i]
    print(f"{i+1:>5d} {hist['loss'][i]:>10.4f} {hist['accuracy'][i]:>10.4f} "
          f"{hist['val_loss'][i]:>10.4f} {hist['val_accuracy'][i]:>10.4f} {lr:>12.6f}")

print(f"\nBest val_loss:     {min(hist['val_loss']):.6f} (epoch {np.argmin(hist['val_loss'])+1})")
print(f"Best val_accuracy: {max(hist['val_accuracy']):.4f} (epoch {np.argmax(hist['val_accuracy'])+1})")

TRAINING HISTORY

Total epochs trained: 13

Epoch       Loss        Acc   Val Loss    Val Acc           LR
────────────────────────────────────────────────────────────
    1     0.4021     0.8164     0.1727     0.9641     0.001000
    2     0.1893     0.9215     0.0752     0.9686     0.001000
    3     0.1324     0.9446     0.3380     0.8952     0.001000
    4     0.1037     0.9588     0.1582     0.9701     0.001000
    5     0.0779     0.9685     0.0569     0.9835     0.001000
    6     0.0629     0.9754     0.0987     0.9641     0.001000
    7     0.0736     0.9714     0.4628     0.8099     0.001000
    8     0.0479     0.9820     0.1043     0.9716     0.001000
    9     0.0363     0.9862     0.0591     0.9850     0.001000
   10     0.0185     0.9935     0.0704     0.9790     0.001000
   11     0.0118     0.9962     0.0661     0.9805     0.001000
   12     0.0092     0.9973     0.0687     0.9850     0.001000
   13     0.0066     0.9980     0.0642     0.9865     0.001000

Best val_los

## 13. Test Set Evaluation

In [18]:
# ============================================================
# TEST SET EVALUATION
# ============================================================

print("=" * 60)
print("TEST EVALUATION (default threshold = 0.5)")
print("=" * 60)

test_loss, test_accuracy = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE, verbose=1)

# Get predictions
probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
predictions = np.argmax(probabilities, axis=1)
p_vaani = probabilities[:, 1]  # P(Vaani) for each sample

# Confusion matrix
confusion = tf.math.confusion_matrix(y_test, predictions, num_classes=NUM_CLASSES).numpy()

tn, fp = confusion[0, 0], confusion[0, 1]
fn, tp = confusion[1, 0], confusion[1, 1]

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

print(f"\nTest loss:     {test_loss:.6f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print()
print("Confusion Matrix:")
print(f"                 Predicted")
print(f"                 Negative  Vaani")
print(f"Actual Negative  {tn:>8d} {fp:>6d}")
print(f"Actual Vaani     {fn:>8d} {tp:>6d}")
print()
print(f"Precision:           {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall / TPR:        {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score:            {f1:.4f}")
print(f"False Positive Rate: {fpr:.4f} ({fpr*100:.2f}%)")
print(f"False Negative Rate: {fnr:.4f} ({fnr*100:.2f}%)")

# Save metrics
test_metrics = {
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "precision": float(precision),
    "recall_tpr": float(recall),
    "f1": float(f1),
    "false_positive_rate": float(fpr),
    "false_negative_rate": float(fnr),
    "true_negative": int(tn),
    "false_positive": int(fp),
    "false_negative": int(fn),
    "true_positive": int(tp),
}

metrics_path = EVALUATION_DIR / "test_metrics_v2.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=4)
print(f"\nSaved: {metrics_path}")

TEST EVALUATION (default threshold = 0.5)
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9877 - loss: 0.0382

Test loss:     0.038202
Test accuracy: 0.9877 (98.77%)

Confusion Matrix:
                 Predicted
                 Negative  Vaani
Actual Negative       637      6
Actual Vaani            2      4

Precision:           0.4000 (40.00%)
Recall / TPR:        0.6667 (66.67%)
F1 Score:            0.5000
False Positive Rate: 0.0093 (0.93%)
False Negative Rate: 0.3333 (33.33%)

Saved: c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\evaluation\test_metrics_v2.json


## 14. Multi-Threshold Evaluation

In [19]:
# ============================================================
# MULTI-THRESHOLD EVALUATION
# ============================================================

THRESHOLDS = [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 0.97, 0.99]

print("=" * 60)
print("MULTI-THRESHOLD EVALUATION")
print("=" * 60)
print()
print(f"{'Thresh':>7s} {'TP':>5s} {'TN':>5s} {'FP':>5s} {'FN':>5s} "
      f"{'Prec':>7s} {'Recall':>7s} {'F1':>7s} {'FPR':>7s} {'FNR':>7s} {'Acc':>7s}")
print("─" * 80)

threshold_results = []

for threshold in THRESHOLDS:
    pred_t = (p_vaani >= threshold).astype(int)

    tp_t = np.sum((pred_t == 1) & (y_test == 1))
    tn_t = np.sum((pred_t == 0) & (y_test == 0))
    fp_t = np.sum((pred_t == 1) & (y_test == 0))
    fn_t = np.sum((pred_t == 0) & (y_test == 1))

    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0.0
    rec_t  = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0.0
    f1_t   = 2 * prec_t * rec_t / (prec_t + rec_t) if (prec_t + rec_t) > 0 else 0.0
    fpr_t  = fp_t / (fp_t + tn_t) if (fp_t + tn_t) > 0 else 0.0
    fnr_t  = fn_t / (fn_t + tp_t) if (fn_t + tp_t) > 0 else 0.0
    acc_t  = (tp_t + tn_t) / (tp_t + tn_t + fp_t + fn_t)

    print(f"{threshold:>7.2f} {tp_t:>5d} {tn_t:>5d} {fp_t:>5d} {fn_t:>5d} "
          f"{prec_t:>7.4f} {rec_t:>7.4f} {f1_t:>7.4f} {fpr_t:>7.4f} {fnr_t:>7.4f} {acc_t:>7.4f}")

    threshold_results.append({
        "threshold": threshold,
        "accuracy": float(acc_t),
        "precision": float(prec_t),
        "recall_tpr": float(rec_t),
        "f1": float(f1_t),
        "fpr": float(fpr_t),
        "fnr": float(fnr_t),
        "tp": int(tp_t), "tn": int(tn_t), "fp": int(fp_t), "fn": int(fn_t),
    })

# Find recommended threshold: Lowest FPR among thresholds with recall >= 95%
viable = [r for r in threshold_results if r["recall_tpr"] >= 0.95]
if viable:
    recommended = min(viable, key=lambda r: r["fpr"])
    rec_thresh = recommended["threshold"]
else:
    recommended = min(threshold_results, key=lambda r: r["fpr"])
    rec_thresh = recommended["threshold"]
    print("\n⚠ No threshold achieves recall >= 95%. Showing lowest FPR overall.")

print(f"\n→ Recommended threshold: {rec_thresh}")
print(f"  Recall: {recommended['recall_tpr']:.4f}, FPR: {recommended['fpr']:.4f}, F1: {recommended['f1']:.4f}")

# Save
threshold_metrics = {
    "model": str(final_model_path),
    "classes": {"0": "negative", "1": "vaani"},
    "preprocessing": {
        "input_range_before_normalization": [-80.0, 0.0],
        "normalization": "(x + 80.0) / 80.0",
        "input_range_after_normalization": [0.0, 1.0],
    },
    "thresholds_evaluated": threshold_results,
    "selection_rule": "Lowest FPR among thresholds with recall >= 95%",
    "recommended_threshold": rec_thresh,
}

thresh_path = EVALUATION_DIR / "threshold_metrics_v2.json"
with open(thresh_path, "w", encoding="utf-8") as f:
    json.dump(threshold_metrics, f, indent=4)
print(f"\nSaved: {thresh_path}")

MULTI-THRESHOLD EVALUATION

 Thresh    TP    TN    FP    FN    Prec  Recall      F1     FPR     FNR     Acc
────────────────────────────────────────────────────────────────────────────────
   0.50     4   637     6     2  0.4000  0.6667  0.5000  0.0093  0.3333  0.9877
   0.60     4   637     6     2  0.4000  0.6667  0.5000  0.0093  0.3333  0.9877
   0.70     4   638     5     2  0.4444  0.6667  0.5333  0.0078  0.3333  0.9892
   0.80     3   639     4     3  0.4286  0.5000  0.4615  0.0062  0.5000  0.9892
   0.85     3   640     3     3  0.5000  0.5000  0.5000  0.0047  0.5000  0.9908
   0.90     3   640     3     3  0.5000  0.5000  0.5000  0.0047  0.5000  0.9908
   0.95     3   640     3     3  0.5000  0.5000  0.5000  0.0047  0.5000  0.9908
   0.97     3   641     2     3  0.6000  0.5000  0.5455  0.0031  0.5000  0.9923
   0.99     3   643     0     3  1.0000  0.5000  0.6667  0.0000  0.5000  0.9954

⚠ No threshold achieves recall >= 95%. Showing lowest FPR overall.

→ Recommended threshol

## 15. Per-Category Negative Evaluation

Uses category metadata saved during feature extraction — NOT fragile filename matching.

**Silence note:** All 307 silence clips originate from a single source recording.
To prevent source leakage, Notebook 01 correctly places ALL silence clips in the training split.
The test set intentionally contains 0 silence clips.

In [20]:
# ============================================================
# PER-CATEGORY NEGATIVE EVALUATION (metadata-based)
# ============================================================

print("=" * 60)
print("PER-CATEGORY FALSE-ACTIVATION ANALYSIS")
print("=" * 60)

neg_categories = ["negative_silence", "negative_background", "negative_speech_commands"]

category_results = {}

for cat in neg_categories:
    # Use saved category metadata — NOT filename matching
    cat_mask = (cats_test == cat)
    n_cat = np.sum(cat_mask)

    if n_cat == 0:
        print(f"\n{cat}:")
        if cat == "negative_silence":
            print("  Silence test evaluation UNAVAILABLE.")
            print("  Reason: All 307 available silence clips originate from a single")
            print("  source recording. To prevent source leakage, Notebook 01 correctly")
            print("  places all silence clips in the training split only.")
            print("  → An independent unseen-room silence stress test is required")
            print("    for final FAR evaluation.")
        else:
            print(f"  No {cat} test samples available.")
        category_results[cat] = {
            "n_clips": 0,
            "status": "unavailable",
            "reason": "all clips in training split (single source recording)"
                      if cat == "negative_silence"
                      else "no test files",
        }
        continue

    cat_probs = p_vaani[cat_mask]

    print(f"\n{cat}:")
    print(f"  Clips: {n_cat}")
    print(f"  Avg P(Vaani):  {np.mean(cat_probs):.6f}")
    print(f"  Max P(Vaani):  {np.max(cat_probs):.6f}")
    print(f"  Min P(Vaani):  {np.min(cat_probs):.6f}")

    for t in [0.50, 0.90, 0.95, 0.99]:
        fires = np.sum(cat_probs >= t)
        pct = fires / n_cat * 100
        print(f"  False fires @ {t:.2f}: {fires:>4d}/{n_cat} ({pct:.1f}%)")

    category_results[cat] = {
        "n_clips": int(n_cat),
        "avg_p_vaani": float(np.mean(cat_probs)),
        "max_p_vaani": float(np.max(cat_probs)),
        "min_p_vaani": float(np.min(cat_probs)),
        "false_fires_050": int(np.sum(cat_probs >= 0.50)),
        "false_fires_090": int(np.sum(cat_probs >= 0.90)),
        "false_fires_095": int(np.sum(cat_probs >= 0.95)),
        "false_fires_099": int(np.sum(cat_probs >= 0.99)),
    }

PER-CATEGORY FALSE-ACTIVATION ANALYSIS

negative_silence:
  Silence test evaluation UNAVAILABLE.
  Reason: All 307 available silence clips originate from a single
  source recording. To prevent source leakage, Notebook 01 correctly
  places all silence clips in the training split only.
  → An independent unseen-room silence stress test is required
    for final FAR evaluation.

negative_background:
  Clips: 119
  Avg P(Vaani):  0.003234
  Max P(Vaani):  0.014537
  Min P(Vaani):  0.000934
  False fires @ 0.50:    0/119 (0.0%)
  False fires @ 0.90:    0/119 (0.0%)
  False fires @ 0.95:    0/119 (0.0%)
  False fires @ 0.99:    0/119 (0.0%)

negative_speech_commands:
  Clips: 524
  Avg P(Vaani):  0.013354
  Max P(Vaani):  0.974460
  Min P(Vaani):  0.000000
  False fires @ 0.50:    6/524 (1.1%)
  False fires @ 0.90:    3/524 (0.6%)
  False fires @ 0.95:    3/524 (0.6%)
  False fires @ 0.99:    0/524 (0.0%)


## 16. False Activation Stress Test

In [21]:
# ============================================================
# FALSE-ACTIVATION STRESS TEST — NEGATIVE-ONLY
# ============================================================

print("=" * 60)
print("FALSE-ACTIVATION STRESS TEST — NEGATIVE-ONLY")
print("=" * 60)

# All negative samples in the test set
neg_mask = (y_test == 0)
neg_probs = p_vaani[neg_mask]
n_neg = np.sum(neg_mask)

print(f"\nTotal negative test samples: {n_neg}")
print(f"\nFAR (False Accept Rate) on independent 1-second clips:")
print(f"{'Threshold':>10s} {'False Accepts':>15s} {'FAR %':>10s}")
print("─" * 40)

stress_results = {}
for t in THRESHOLDS:
    fa = np.sum(neg_probs >= t)
    far = fa / n_neg * 100 if n_neg > 0 else 0
    print(f"{t:>10.2f} {fa:>15d} {far:>10.2f}%")
    stress_results[str(t)] = {"false_accepts": int(fa), "far_pct": float(far)}

print(f"\n{'─' * 50}")
print("Per-category breakdown at threshold 0.90:")
for cat, res in category_results.items():
    if res.get("n_clips", 0) == 0:
        status = res.get("reason", "no test samples")
        print(f"  {cat:<30s}: N/A ({status})")
    else:
        n = res["n_clips"]
        fa = res["false_fires_090"]
        pct = fa / n * 100 if n > 0 else 0
        print(f"  {cat:<30s}: {fa:>4d}/{n} ({pct:.1f}%)")

# Combined
fa_combined = np.sum(neg_probs >= 0.90)
far_combined = fa_combined / n_neg * 100 if n_neg > 0 else 0
print(f"  {'Combined negatives':<30s}: {fa_combined:>4d}/{n_neg} ({far_combined:.1f}%)")

print(f"\n{'─' * 50}")
print("FALSE ACTIVATIONS / HOUR:")
print("  Cannot compute from independent 1-second clips.")
print("  This metric requires continuous audio recordings.")
print("  A continuous-audio test will be conducted separately.")

# Save
stress_path = EVALUATION_DIR / "false_activation_analysis_v2.json"
with open(stress_path, "w", encoding="utf-8") as f:
    json.dump({
        "total_negative_test_samples": int(n_neg),
        "threshold_results": stress_results,
        "per_category": category_results,
        "note": "FAR computed on independent 1-second clips. "
                "Hourly false-activation rate requires continuous recordings.",
        "silence_note": "All silence clips are in training only (single source recording). "
                        "Unseen-room silence stress test required.",
    }, f, indent=4)
print(f"\nSaved: {stress_path}")

FALSE-ACTIVATION STRESS TEST — NEGATIVE-ONLY

Total negative test samples: 643

FAR (False Accept Rate) on independent 1-second clips:
 Threshold   False Accepts      FAR %
────────────────────────────────────────
      0.50               6       0.93%
      0.60               6       0.93%
      0.70               5       0.78%
      0.80               4       0.62%
      0.85               3       0.47%
      0.90               3       0.47%
      0.95               3       0.47%
      0.97               2       0.31%
      0.99               0       0.00%

──────────────────────────────────────────────────
Per-category breakdown at threshold 0.90:
  negative_silence              : N/A (all clips in training split (single source recording))
  negative_background           :    0/119 (0.0%)
  negative_speech_commands      :    3/524 (0.6%)
  Combined negatives            :    3/643 (0.5%)

──────────────────────────────────────────────────
FALSE ACTIVATIONS / HOUR:
  Cannot compute fr

## 17. V1 vs V2 Comparison

V1 and V2 use the **same DS-CNN-Small architecture** and the **same Log-Mel feature frontend**.
This comparison isolates the effect of the improved V2 dataset.

In [22]:
# ============================================================
# V1 vs V2 COMPARISON
# ============================================================

print("=" * 60)
print("V1 vs V2 COMPARISON")
print("=" * 60)
print()
print("Note: V1 and V2 use identical DS-CNN architecture and feature frontend.")
print("This comparison isolates the effect of dataset changes.")

v1_metrics = None
v1_thresholds = None

if V1_METRICS_PATH.exists():
    with open(V1_METRICS_PATH, "r") as f:
        v1_metrics = json.load(f)
    print(f"\nLoaded V1 metrics from: {V1_METRICS_PATH}")

if V1_THRESHOLD_PATH.exists():
    with open(V1_THRESHOLD_PATH, "r") as f:
        v1_thresholds = json.load(f)
    print(f"Loaded V1 thresholds from: {V1_THRESHOLD_PATH}")

if v1_metrics:
    print()
    print(f"{'Metric':<25s} {'V1':>12s} {'V2':>12s} {'Change':>12s}")
    print("─" * 65)

    comparisons = [
        ("Accuracy", "test_accuracy", test_metrics["test_accuracy"]),
        ("Precision", "precision", test_metrics["precision"]),
        ("Recall / TPR", "recall_tpr", test_metrics["recall_tpr"]),
        ("F1 Score", "f1", test_metrics["f1"]),
        ("FPR", "false_positive_rate", test_metrics["false_positive_rate"]),
        ("FNR", "false_negative_rate", test_metrics["false_negative_rate"]),
    ]

    for label, v1_key, v2_val in comparisons:
        v1_val = v1_metrics.get(v1_key, None)
        if v1_val is not None:
            change = v2_val - v1_val
            arrow = "↑" if change > 0 else "↓" if change < 0 else "="
            if "rate" in v1_key.lower() or "fnr" in label.lower() or "fpr" in label.lower():
                quality = "✓" if change <= 0 else "✗"
            else:
                quality = "✓" if change >= 0 else "✗"
            print(f"{label:<25s} {v1_val:>12.4f} {v2_val:>12.4f} {change:>+10.4f} {arrow} {quality}")
        else:
            print(f"{label:<25s} {'N/A':>12s} {v2_val:>12.4f}")

    # Threshold 0.90 comparison
    if v1_thresholds:
        v1_at_090 = None
        for t in v1_thresholds.get("thresholds_evaluated", []):
            if abs(t["threshold"] - 0.90) < 0.001:
                v1_at_090 = t
                break

        v2_at_090 = None
        for t in threshold_results:
            if abs(t["threshold"] - 0.90) < 0.001:
                v2_at_090 = t
                break

        if v1_at_090 and v2_at_090:
            print()
            print("At threshold = 0.90:")
            print(f"{'Metric':<25s} {'V1':>12s} {'V2':>12s}")
            print("─" * 50)
            for key in ["accuracy", "precision", "recall_tpr", "f1", "fpr", "fnr"]:
                v1_v = v1_at_090.get(key, None)
                v2_v = v2_at_090.get(key, None)
                if v1_v is not None and v2_v is not None:
                    print(f"{key:<25s} {v1_v:>12.4f} {v2_v:>12.4f}")
else:
    print("\n⚠ V1 metrics not found. Skipping comparison.")
    print(f"  Expected: {V1_METRICS_PATH}")

V1 vs V2 COMPARISON

Note: V1 and V2 use identical DS-CNN architecture and feature frontend.
This comparison isolates the effect of dataset changes.

Loaded V1 metrics from: c:\Users\Mayank Singh\Codes\SIH 2026\model_v1\evaluation\test_metrics.json
Loaded V1 thresholds from: c:\Users\Mayank Singh\Codes\SIH 2026\model_v1\evaluation\threshold_metrics.json

Metric                              V1           V2       Change
─────────────────────────────────────────────────────────────────
Accuracy                        0.9679       0.9877    +0.0198 ↑ ✓
Precision                       0.9561       0.4000    -0.5561 ↓ ✗
Recall / TPR                    0.9820       0.6667    -0.3153 ↓ ✗
F1 Score                        0.9689       0.5000    -0.4689 ↓ ✗
FPR                             0.0467       0.0093    -0.0374 ↓ ✓
FNR                             0.0180       0.3333    +0.3153 ↑ ✗

At threshold = 0.90:
Metric                              V1           V2
────────────────────────────────────

In [23]:
# ============================================================
# SAVE V1 vs V2 COMPARISON
# ============================================================

comparison = {
    "v2_metrics": test_metrics,
    "v2_threshold_090": next((t for t in threshold_results if abs(t["threshold"] - 0.90) < 0.001), None),
    "v2_recommended_threshold": rec_thresh,
    "note": "V1 and V2 use identical DS-CNN architecture and feature frontend. "
            "Comparison isolates dataset effect.",
}

if v1_metrics:
    comparison["v1_metrics"] = v1_metrics
if v1_thresholds:
    v1_at_090 = next((t for t in v1_thresholds.get("thresholds_evaluated", []) if abs(t["threshold"] - 0.90) < 0.001), None)
    if v1_at_090:
        comparison["v1_threshold_090"] = v1_at_090

comp_path = EVALUATION_DIR / "v1_v2_comparison.json"
with open(comp_path, "w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=4)
print(f"Saved comparison: {comp_path}")

Saved comparison: c:\Users\Mayank Singh\Codes\SIH 2026\model_v2\evaluation\v1_v2_comparison.json


## 18. Final Summary

In [24]:
# ============================================================
# CLEANUP TEMP FILES
# ============================================================

if TEMP_DIR.exists():
    temp_files = list(TEMP_DIR.iterdir())
    if temp_files:
        shutil.rmtree(str(TEMP_DIR))
        TEMP_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Cleaned {len(temp_files)} temp files.")
    else:
        print("Temp directory empty.")

# ============================================================
# FINAL SUMMARY
# ============================================================

# Count original vs augmented training files
n_orig_pos_train = len(dataset_files[("train", "positive")])
n_aug_pos_train = int(np.sum((cats_train == "positive") & np.array(["train_augmented" in str(p) for p in paths_train])))
n_orig_neg_train = sum(len(dataset_files[("train", cat)]) for cat in CATEGORIES if cat != "positive")
n_aug_neg_train = int(np.sum((cats_train != "positive") & np.array(["train_augmented" in str(p) for p in paths_train])))

print()
print("=" * 70)
print("  V2 TRAINING & EVALUATION — FINAL SUMMARY")
print("=" * 70)

print(f"\n{'─' * 70}")
print("DATASET")
print(f"  TRAIN:")
print(f"    original positive:        {n_orig_pos_train:>6,}")
print(f"    augmented positive:       {n_aug_pos_train:>6,}")
print(f"    original negative:        {n_orig_neg_train:>6,}")
print(f"    augmented negative:       {n_aug_neg_train:>6,}")
print(f"    total:                    {len(y_train):>6,}")
print(f"  VALIDATION:")
print(f"    positive:                 {np.sum(y_val==1):>6,}")
print(f"    negative:                 {np.sum(y_val==0):>6,}")
print(f"    total:                    {len(y_val):>6,}")
print(f"  TEST:")
print(f"    positive:                 {np.sum(y_test==1):>6,}")
print(f"    negative:                 {np.sum(y_test==0):>6,}")
print(f"    total:                    {len(y_test):>6,}")

print(f"\n{'─' * 70}")
print("FEATURES")
print(f"  train shape:      {X_train.shape}")
print(f"  validation shape: {X_val.shape}")
print(f"  test shape:       {X_test.shape}")

print(f"\n{'─' * 70}")
print("MODEL")
print(f"  Architecture:     DS-CNN-Small (V1 baseline, UNCHANGED)")
print(f"  Parameters:       {total_params:,}")
print(f"  FP32 size:        {fp32_size_kb:.2f} KB")
print(f"  Input shape:      {INPUT_SHAPE} (derived dynamically)")

print(f"\n{'─' * 70}")
print("EVALUATION (threshold = 0.50)")
print(f"  Accuracy:         {test_metrics['test_accuracy']:.4f} ({test_metrics['test_accuracy']*100:.2f}%)")
print(f"  Precision:        {test_metrics['precision']:.4f}")
print(f"  Recall:           {test_metrics['recall_tpr']:.4f}")
print(f"  F1:               {test_metrics['f1']:.4f}")
print(f"  FPR:              {test_metrics['false_positive_rate']:.4f} ({test_metrics['false_positive_rate']*100:.2f}%)")
print(f"  FNR:              {test_metrics['false_negative_rate']:.4f}")

print(f"\n{'─' * 70}")
print(f"RECOMMENDED THRESHOLD: {rec_thresh}")
rec_result = next((t for t in threshold_results if abs(t["threshold"] - rec_thresh) < 0.001), None)
if rec_result:
    print(f"  Recall:  {rec_result['recall_tpr']:.4f}")
    print(f"  FPR:     {rec_result['fpr']:.4f}")
    print(f"  F1:      {rec_result['f1']:.4f}")

print(f"\n{'─' * 70}")
print("SILENCE STATUS")
n_silence_test = int(np.sum(cats_test == "negative_silence"))
if n_silence_test == 0:
    print("  Silence clips are training-only because the available silence dataset")
    print("  originates from a single source recording. An independent unseen-room")
    print("  silence stress test is required for final FAR evaluation.")
else:
    sil_probs = p_vaani[cats_test == "negative_silence"]
    print(f"  Silence test clips: {n_silence_test}")
    print(f"  Avg P(Vaani): {np.mean(sil_probs):.6f}")

print(f"\n{'─' * 70}")
print("IMPORTANT")
print("  The primary goal is REDUCING FALSE ACTIVATIONS while maintaining high recall.")
print("  Do NOT evaluate success based only on accuracy.")
print("  Quantization/TFLite conversion will be done in a separate step.")

print(f"\n{'─' * 70}")
print("OUTPUTS SAVED")
print(f"  Features:         {FEATURE_DIR}")
print(f"  Best checkpoint:  {best_model_path}")
print(f"  Final model:      {final_model_path}")
print(f"  Model config:     {config_path}")
print(f"  Test metrics:     {metrics_path}")
print(f"  Threshold eval:   {thresh_path}")
print(f"  Training history: {EVALUATION_DIR / 'training_history_v2.csv'}")
print(f"  Comparison:       {comp_path}")
print(f"  Aug manifest:     {aug_manifest_path}")

print()
print("=" * 70)
print("  V2 TRAINING COMPLETE")
print("=" * 70)

Temp directory empty.

  V2 TRAINING & EVALUATION — FINAL SUMMARY

──────────────────────────────────────────────────────────────────────
DATASET
  TRAIN:
    original positive:           322
    augmented positive:          644
    original negative:         3,307
    augmented negative:        6,614
    total:                    10,887
  VALIDATION:
    positive:                     24
    negative:                    644
    total:                       668
  TEST:
    positive:                      6
    negative:                    643
    total:                       649

──────────────────────────────────────────────────────────────────────
FEATURES
  train shape:      (10887, 40, 49, 1)
  validation shape: (668, 40, 49, 1)
  test shape:       (649, 40, 49, 1)

──────────────────────────────────────────────────────────────────────
MODEL
  Architecture:     DS-CNN-Small (V1 baseline, UNCHANGED)
  Parameters:       5,324
  FP32 size:        20.80 KB
  Input shape:      (40, 49, 1)